# pymefpp_partition_calibrations

Notebook to create a partition of the calibration files for later reduction

In [ ]:
# partition list of calibration frames
from os import listdir
from os.path import isfile, join
import glob
from astropy.io import fits
import math
import operator
import copy
import datetime
import ephem
import os
from pathlib import Path

In [ ]:
class Calibration:
    
    def __init__(self, filename):

        self.filename = filename
        self.dpr_type = getKeyVal(filename, 'HIERARCH ESO DPR TYPE')
        self.obsid = getKeyVal(filename, 'HIERARCH ESO OBS ID')
        self.OBstart = getKeyVal(filename, 'HIERARCH ESO OBS START')
        self.tplnexp = getKeyVal(filename, 'HIERARCH ESO TPL NEXP')
        self.tplexpno = getKeyVal(filename, 'HIERARCH ESO TPL EXPNO')
        self.filter = getKeyVal(filename, 'HIERARCH ESO INS FILT1 NAME')
        self.exptime = getKeyVal(filename, 'EXPTIME')
        
class BiasPartition:
        
    def __init__(self):

        self.numberList = []
        self.partList = []                    
        
    def insertCal(self, bias):
        createNewPartition = True
        if len(self.partList) > 0:
            for partition in self.partList:
                if partition[0].OBstart == bias.OBstart:
                    partition.append(bias)
                    createNewPartition = False
                else:
                    pass
        if createNewPartition:
            self.partList.append([bias])
            self.numberList.append(len(self.partList))
            
class FlatPartition:
        
    def __init__(self):

        self.numberList = []
        self.partList = []                    
        
    def insertCal(self, flat):
        createNewPartition = True
        if len(self.partList) > 0:
            for partition in self.partList:
                if partition[0].filter == flat.filter and \
                         partition[0].dpr_type == flat.dpr_type and \
                         partition[0].OBstart == flat.OBstart:
                    partition.append(flat)
                    createNewPartition = False
                else:
                    pass
        if createNewPartition:
            self.partList.append([flat])
            self.numberList.append(len(self.partList))
            
def getKeyVal(filename, key):
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        val = primary_header[key]
        return val

In [ ]:
# we set the data reduction directories
raw_data_dir = '/home/lschmidt/fernando/fits/'

reduced_cal_dir = '/home/fselman/omegacam/fernando/reduced/calibs/'
try:
    os.mkdir(reduced_cal_dir)
except:
    pass

In [ ]:
listing = glob.glob(raw_data_dir+'OMEGA.*.fits')

print('%d OmegaCAM frames in %s'%(len(listing), raw_data_dir))

biasPartition = BiasPartition()
domePartition = FlatPartition()
skyPartition = FlatPartition()
bias_list = []
dome_flat_list = []
sky_flat_list = []
i_bias = 0
i_dome = 0
i_sky = 0
i_else = 0
for file_name in listing:
    # open FITS and read headers
    hdu_list = fits.open(file_name)
    primary_header = hdu_list[0].header
    dpr_catg = primary_header['HIERARCH ESO DPR CATG']
    dpr_type = primary_header['HIERARCH ESO DPR TYPE']
    hdu_list.close()
    if dpr_catg == 'CALIB' and dpr_type == 'BIAS':
        bias = Calibration(file_name)
        i_bias += 1
        if i_bias%10:
            print("Reading BIAS frame: %d" % i_bias, end='\r')
        biasPartition.insertCal(bias)
        bias_list.append(bias)
        
    if dpr_catg == 'CALIB' and dpr_type == 'FLAT,DOME':
        dome_flat = Calibration(file_name)
        i_dome += 1
        if i_dome%10:
            print("Reading FLAT,DOME frame: %d" % i_dome, end='\r')
        domePartition.insertCal(dome_flat)
        dome_flat_list.append(dome_flat)
        
    if dpr_catg == 'CALIB' and dpr_type == 'FLAT,SKY':
        sky_flat = Calibration(file_name)
        i_sky += 1
        if i_sky%10:
            print("Reading FLAT,SKY frame: %d" % i_sky, end='\r')
        skyPartition.insertCal(sky_flat)
        sky_flat_list.append(sky_flat)
    else:
        i_else+= 1
        print("Reading frame: %s %s %d" % (dpr_catg, dpr_type, i_sky,), end='\r')    
    

    
print('%d OmegaCAM BIAS frames in %s'%(len(bias_list), raw_data_dir))   
print("%d partitions"%(len(biasPartition.numberList)))

print('%d OmegaCAM DOME,FLAT frames in %s'%(len(dome_flat_list), raw_data_dir))   
print("%d partitions"%(len(domePartition.numberList)))

print('%d OmegaCAM SKY,FLAT frames in %s'%(len(sky_flat_list), raw_data_dir))   
print("%d partitions"%(len(skyPartition.numberList)))

print("%d non included frames"%(i_else,))

In [ ]:
# we create directory structure to reduce the calibration data

# bad pixel masks BPM are needed but static at this point we create
# a directory to contain them
try:
    os.makedirs(reduced_cal_dir+'bpm')
except:
    pass

Path(reduced_cal_dir+'bpm'+'/bpm.list').touch()
Path(reduced_cal_dir+'bpm'+'/toreduce_bpm').touch()

# ... here we deal with the biases
for partition in biasPartition.partList:
    firstBias = partition[0]
    print(firstBias.OBstart, firstBias.filter, len(partition))
    biasList = []
    for bias in partition:
        biasList.append([bias.OBstart, bias.tplexpno, bias.tplnexp, bias.obsid, bias.filename])
        #print(obsList)
    newList = copy.deepcopy(sorted(biasList, key = operator.itemgetter(0, 1)))
    #print(newList)
    for aBias in newList:
        subdir_str = 'bias/'+"%d"%aBias[3]+'/'+aBias[0]
        print(subdir_str)
        try:
            os.makedirs(reduced_cal_dir+subdir_str)
        except:
            pass
        
        try:
            os.chdir(reduced_cal_dir+subdir_str)
            with open('bias.list', 'a') as f:
                print(aBias[4], file=f)
            os.chdir(reduced_cal_dir)
        except:
            raise Exception('could not change directory to: '+reduced_cal_dir+subdir_str)
            


In [ ]:
# ... and next we deal with the dome flats
for partition in domePartition.partList:
    firstDome = partition[0]
    print(firstDome.OBstart, firstDome.filter, len(partition))
    domeList = []
    for dome in partition:
        domeList.append([dome.OBstart, dome.tplexpno, dome.tplnexp, dome.obsid, dome.filename])
        #print(obsList)
    newList = copy.deepcopy(sorted(domeList, key = operator.itemgetter(0, 1)))
    #print(newList)
    for aDome in newList:
        subdir_str = 'dome/'+"%d"%aDome[3]+'/'+firstDome.filter+'/'+aDome[0]
        print(subdir_str)
        try:
            os.makedirs(reduced_cal_dir+subdir_str)
        except:
            pass
        
        try:
            os.chdir(reduced_cal_dir+subdir_str)
            with open('dome.list', 'a') as f:
                print(aDome[4], file=f)
            os.chdir(reduced_cal_dir)
        except:
            raise Exception('could not change directory to: '+reduced_cal_dir+subdir_str)
            

In [ ]:
# ... and last we deal with the sky flats
for partition in skyPartition.partList:
    firstSky = partition[0]
    print(firstSky.OBstart, firstSky.filter, len(partition))
    skyList = []
    for sky in partition:
        skyList.append([sky.OBstart, sky.tplexpno, sky.tplnexp, sky.obsid, sky.filename])
        #print(skyList)
    newList = copy.deepcopy(sorted(skyList, key = operator.itemgetter(0, 1)))
    #print(newList)
    for aSky in newList:
        subdir_str = 'sky/'+"%d"%aSky[3]+'/'+firstSky.filter+'/'+aSky[0]
        print(subdir_str)
        try:
            os.makedirs(reduced_cal_dir+subdir_str)
        except:
            pass
        
        try:
            os.chdir(reduced_cal_dir+subdir_str)
            with open('sky.list', 'a') as f:
                print(aDome[4], file=f)
            os.chdir(reduced_cal_dir)
        except:
            raise Exception('could not change directory to: '+reduced_cal_dir+subdir_str)
            